# Silver processing

Esta notebook construye la capa Silver a partir de Bronze.

Responsabilidades:

- Normalizar campos.
- Derivar columnas base de negocio.
- Calcular monto total.
- Identificar operaciones de fin de semana.
- Marcar outliers de monto sin eliminar registros.


In [0]:
from pyspark.sql import functions as F

In [0]:
BRONZE_TABLE = "byma.bronze.operaciones_raw"
SILVER_TABLE = "byma.silver.operaciones_clean"

In [0]:
df_bronze = spark.table(BRONZE_TABLE)

display(df_bronze.limit(10))
df_bronze.printSchema()

fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,_ingested_at,_source_file,_pipeline_version,fecha_partition
2026-01-08T00:36:46.000Z,Compra,CLI99F147FF,Grupo Financiero Galicia S.A,ARS,GGAL,2.000000,8387.630400,TXNe1v2isqp49kwv,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T01:59:41.000Z,Compra,CLI1499FE7A,Ternium Argentina Sa,ARS,TXAR,50.000000,750.218400,TXNnb74x2ek3zezq,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T02:34:15.000Z,Compra,CLICD83AF92,Cedear Apple Inc.,ARS,AAPL,1.000000,20248.747100,TXN0153sflse45bg,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T04:34:00.000Z,Venta,CLI1179013D,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,450.000000,1003.148900,TXNou6qlz2rwgwaj,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T12:13:10.000Z,Compra,CLIE6D54A7A,Bono Rep. Argentina Usd Step Up 2030,USD,AL30D,1334.000000,0.671300,TXNdw79wcxz2ei92,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T11:06:53.000Z,Compra,CLIE4A0B9FA,Cedear Pan American Silver Cor,ARS,PAAS,1.000000,26379.030800,TXNgvz9aawe429om,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T09:50:59.000Z,Compra,CLIC6141084,Banco Macro,ARS,BMA,10.000000,13985.177300,TXNr08cw2lbm2qsq,Sitio Web Desktop,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T18:13:36.000Z,Compra,CLI134F00C3,Cedear Apple Inc.,ARS,AAPL,12.000000,20070.758200,TXNt8cq1xgbngz6w,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T00:47:50.000Z,Compra,CLIBA20C83C,"Cedear Amazon.Com, Inc",ARS,AMZN,27.000000,2576.639900,TXN6l91f4x2bfibj,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08
2026-01-08T11:41:18.000Z,Venta,CLIA68BD402,Cedear The Coca Cola Company,ARS,KO,1.000000,20703.819000,TXN2tnxhn5e0ev6x,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08


root
 |-- fecha: timestamp (nullable = true)
 |-- tipoTran: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- descripcion_titulo: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- simbolo_titulo: string (nullable = true)
 |-- cantidad: decimal(20,6) (nullable = true)
 |-- precio: decimal(20,6) (nullable = true)
 |-- id_transaccion: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _pipeline_version: string (nullable = true)
 |-- fecha_partition: date (nullable = true)



## Transformaciones Silver

In [0]:
df_silver = (
    df_bronze
    .withColumn("fecha_operacion", F.to_date("fecha"))
    .withColumn("monto_total", F.col("cantidad") * F.col("precio"))
    .withColumn("es_fin_de_semana", F.dayofweek("fecha").isin([1, 7]))
)
display(df_silver.limit(10))

fecha,tipoTran,id_cliente,descripcion_titulo,moneda,simbolo_titulo,cantidad,precio,id_transaccion,origen,_ingested_at,_source_file,_pipeline_version,fecha_partition,fecha_operacion,monto_total,es_fin_de_semana
2026-01-08T00:36:46.000Z,Compra,CLI99F147FF,Grupo Financiero Galicia S.A,ARS,GGAL,2.000000,8387.630400,TXNe1v2isqp49kwv,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,16775.260800000,false
2026-01-08T01:59:41.000Z,Compra,CLI1499FE7A,Ternium Argentina Sa,ARS,TXAR,50.000000,750.218400,TXNnb74x2ek3zezq,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,37510.920000000,false
2026-01-08T02:34:15.000Z,Compra,CLICD83AF92,Cedear Apple Inc.,ARS,AAPL,1.000000,20248.747100,TXN0153sflse45bg,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,20248.747100000,false
2026-01-08T04:34:00.000Z,Venta,CLI1179013D,Bono Rep. Argentina Usd Step Up 2030,ARS,AL30,450.000000,1003.148900,TXNou6qlz2rwgwaj,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,451417.005000000,false
2026-01-08T12:13:10.000Z,Compra,CLIE6D54A7A,Bono Rep. Argentina Usd Step Up 2030,USD,AL30D,1334.000000,0.671300,TXNdw79wcxz2ei92,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,895.514200000,false
2026-01-08T11:06:53.000Z,Compra,CLIE4A0B9FA,Cedear Pan American Silver Cor,ARS,PAAS,1.000000,26379.030800,TXNgvz9aawe429om,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,26379.030800000,false
2026-01-08T09:50:59.000Z,Compra,CLIC6141084,Banco Macro,ARS,BMA,10.000000,13985.177300,TXNr08cw2lbm2qsq,Sitio Web Desktop,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,139851.773000000,false
2026-01-08T18:13:36.000Z,Compra,CLI134F00C3,Cedear Apple Inc.,ARS,AAPL,12.000000,20070.758200,TXNt8cq1xgbngz6w,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,240849.098400000,false
2026-01-08T00:47:50.000Z,Compra,CLIBA20C83C,"Cedear Amazon.Com, Inc",ARS,AMZN,27.000000,2576.639900,TXN6l91f4x2bfibj,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,69569.277300000,false
2026-01-08T11:41:18.000Z,Venta,CLIA68BD402,Cedear The Coca Cola Company,ARS,KO,1.000000,20703.819000,TXN2tnxhn5e0ev6x,App Mobile,2026-06-27T22:01:11.245Z,/Volumes/byma/bronze/landing/data_set_challenge.csv,1.0.0,2026-01-08,2026-01-08,20703.819000000,false


## Outlier detection

In [0]:
outlier_threshold = df_silver.approxQuantile("monto_total", [0.999], 0.01)[0]

df_silver = df_silver.withColumn(
    "flag_outlier_monto",
    F.col("monto_total") > F.lit(outlier_threshold)
)

print(f"Umbral outlier monto_total p99.9: {outlier_threshold}")

display(
    df_silver
    .select(
        "id_transaccion",
        "simbolo_titulo",
        "cantidad",
        "precio",
        "monto_total",
        "flag_outlier_monto"
    )
    .orderBy(F.col("monto_total").desc())
    .limit(20)
)

Umbral outlier monto_total p99.9: 1472943883.3148


id_transaccion,simbolo_titulo,cantidad,precio,monto_total,flag_outlier_monto
TXNc0opn4b5ng78w,TTS26,1067195974.000000,1.380200,1472943883.314800000,false
TXN9xv9rxus0yesn,T30J6,976270797.000000,1.255000,1225219850.235000000,false
TXNfyfdnr8r9jmfe,T13F6,661750591.000000,1.423800,942200491.465800000,false
TXNqyqpf7xhq7b92,T13F6,496675425.000000,1.418900,704732760.532500000,false
TXNrk37m9n7bx9ml,T13F6,289899750.000000,1.402900,406700359.275000000,false
TXNa26ztf6txdux5,TZXM6,143927228.000000,2.067500,297569543.890000000,false
TXNk2jc8v8ux93qr,SPY,4267.000000,52534.668100,224165428.782700000,false
TXN4zl101astev18,S17A6,213473214.000000,1.034100,220752650.597400000,false
TXNngjodukfkzdtr,TRAN,50977.000000,3757.538700,191548050.309900000,false
TXNxgm99t7d19bf8,S17A6,171938257.000000,1.026100,176425845.507700000,false


## Persist Silver

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS byma.silver")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

silver_count = spark.table(SILVER_TABLE).count()

print(f"Registros persistidos en Silver: {silver_count}")



Registros persistidos en Silver: 100000
